# 🦕 trikedb quickstart

**The single-file graph database.** A knowledge graph in one YAML file — full SPARQL 1.1, built for LLM agents.

This notebook starts where the facts actually start: an ordinary document. It declares a vocabulary, builds an extraction prompt out of the graph itself, imports what a model answered, queries it with SPARQL, and renders the graph **inline**.

In [ ]:
%pip install -q trikedb

## 1. The document

Nothing special about it — a memo someone wrote for people to read. This is the input; everything below is derived from it.

In [ ]:
document = """# Acme ingestion review — Q3 2026

| version | date       | change                            |
|---------|------------|-----------------------------------|
| 1.1     | 2026-05-12 | first circulated to the platform team |
| 1.2     | 2026-06-30 | adds the Figly section            |

## Vendors and the jobs they feed

SalesFlow CRM provides crm-sync-job, which runs hourly and lands in
RAW_CRM_CONTACTS. Contact rows carry personal data, so the table is
restricted.

Adastra Ads provides ads-spend-collector. That job writes
RAW_AD_SPEND_DAILY every morning at 06:00.

Figly was signed in June and provides figly-export-job. Nothing is
scheduled for it yet — the first run is planned for Q4.

## Migration off the legacy table

LEGACY_CONTACTS was retired this quarter. Everything that used to read it
now reads RAW_CRM_CONTACTS instead.

## Changes we had to absorb

RAW_AD_SPEND_DAILY was affected by the Adastra API v3 switch on
2026-04-01: spend arrives in micros now, and the table's numbers jumped by
a factor of a million for a day before anyone noticed.
"""

open("ingestion_review.md", "w").write(document)
print(document)

## 2. Declare the vocabulary

The ontology is the list of predicates a fact is allowed to use. Declaring it before any extraction is the whole idea: a model that is handed the vocabulary cannot answer with a predicate you would have had to clean up afterwards.

In [ ]:
from trikedb import TrikeDB, OntologyError

db = TrikeDB("pipeline.yaml", ontology={
    "PROVIDES":    "SaaS vendor -> ingestion job",
    "INGESTS_TO":  "ingestion job -> warehouse table",
    "MIGRATED_TO": "deprecated table -> its replacement",
    "AFFECTED_BY": "table -> change event",
})
db.save()
db

## 3. The prompt is built from the graph

`extract_prompt` assembles the prompt out of this graph's own declared predicates and the node names already in it. trikedb never calls a model and never reads an API key — you get a string, and you send it wherever you already send prompts.

In [ ]:
prompt = db.extract_prompt(document)

print(len(prompt), "characters")
print(prompt[:prompt.index("## What counts as a fact")])

It also tells the model what *not* to extract. The head of a document is the most prominent string in it, so it is the thing a model hands back as an entity — and a graph that grows a node for a *file*, standing among the people and systems the file is about, cannot be repaired afterwards. trikedb reads the head and names it in the prompt:

In [ ]:
from trikedb.importers import document_title, graph_filename

head = document_title(document)
print("head    :", head)
print("as a file:", graph_filename(head))   # if you keep this document's facts on their own
print()
print(prompt[prompt.index("## The document is not one of the facts"): prompt.index("## Output")])

## 4. What the model answered

Any model, any SDK. The answer is a Markdown table with `s`/`p`/`o` columns — the same format the importer already reads, so a model's output and a table someone typed into a design doc travel the same path.

```python
# live, with whatever you already pay for — see examples/extract_providers.py
rows = db.extract(document, llm=anthropic())
```

So that this notebook runs with no key and no network, here is what one answered, pasted verbatim:

In [ ]:
answer = """| s | p | o | at | prov |
|---|---|---|---|---|
| SalesFlow CRM | PROVIDES | crm-sync-job | | "SalesFlow CRM provides crm-sync-job" |
| crm-sync-job | INGESTS_TO | RAW_CRM_CONTACTS | | "lands in RAW_CRM_CONTACTS" |
| Adastra Ads | PROVIDES | ads-spend-collector | | "Adastra Ads provides ads-spend-collector" |
| ads-spend-collector | INGESTS_TO | RAW_AD_SPEND_DAILY | | "That job writes RAW_AD_SPEND_DAILY" |
| Figly | PROVIDES | figly-export-job | | "Figly ... provides figly-export-job" |
| LEGACY_CONTACTS | MIGRATED_TO | RAW_CRM_CONTACTS | | "Everything that used to read it now reads RAW_CRM_CONTACTS instead" |
| RAW_AD_SPEND_DAILY | AFFECTED_BY | Adastra API v3 switch | 2026-04-01 | "RAW_AD_SPEND_DAILY was affected by the Adastra API v3 switch on 2026-04-01" |
"""

open("facts.md", "w").write(answer)
print(answer)

## 5. Read it before you keep it

`preview` says what each row would do to the graph — new, same, an update, a conflict with a fact already in it, or rejected by the ontology. Nothing is written yet.

In [ ]:
rows = db.read_file("facts.md")

for f in db.preview(rows):
    print(f"{f['verdict']:9} {f['triple']}  {f.get('detail', '')}")

In [ ]:
print("imported:", db.import_file("facts.md"), "triples")
db.save()
db

## 6. What kind of thing each one is

The model gave you the edges. What a node *is* — a vendor, a job, a table — is yours to say, and it is worth saying: `type` drives the colours in the view, and everything else on a node is free-form and travels with it.

In [ ]:
for vendor in db.subjects("PROVIDES"):
    db.set_node(vendor, type="saas")
for job in db.objects(p="PROVIDES"):
    db.set_node(job, type="job")
for table in db.objects(p="INGESTS_TO") + db.subjects("MIGRATED_TO"):
    db.set_node(table, type="table")

db.set_node("SalesFlow CRM", label="SalesFlow CRM", url="https://salesflow.example")
db.set_node("RAW_CRM_CONTACTS", pii=True)   # the document said so; the ontology has no predicate for it
db.set_node("Adastra API v3 switch", type="event")
db.save()
db.node("RAW_CRM_CONTACTS")

The ontology guard is not advice — a hallucinated predicate is rejected on the way in, whether it comes from a model or from you:

In [ ]:
try:
    db.add("crm-sync-job", "TOTALLY_MADE_UP", "x")
except OntologyError as e:
    print("rejected:", e)

And the document itself did not become a node. The head, the headings and the revision-history rows are the paper, not the facts:

In [ ]:
print([t.spo() for t in db if head in (t.s, t.o)])
print([t.spo() for t in db if "1.2" in t.o])

## 7. Query it

Zero-dependency pattern matching, or real SPARQL 1.1 (rdflib engine). Writes go through SPARQL too and land back in the YAML.

In [ ]:
db.query(["?vendor PROVIDES ?job", "?job INGESTS_TO ?table"])

In [ ]:
db.sparql("""
  SELECT ?vendor ?table WHERE {
    ?vendor t:PROVIDES ?job .
    ?job t:INGESTS_TO ?table .
  }
""")

Writes go through SPARQL too, and land back in the YAML:

In [ ]:
db.sparql("INSERT DATA { t:figly-export-job t:INGESTS_TO t:RAW_FIGLY_EXPORTS }")
db.save()
print(db.sparql("ASK { t:figly-export-job t:INGESTS_TO ?table }"))

## 8. The next document knows what the first one said

The prompt is rebuilt from the graph every time, so the entities that just landed are now in it — which is what stops the second document from introducing `RAW_CRM_CONTACTS (v2)` as a second name for a table the graph already has.

In [ ]:
later = db.extract_prompt("Figly's first export ran on 2026-10-02.")

print(later[later.index("## Entities that already exist"): later.index("## What counts as a fact")])

## 9. See the graph — right here in the notebook

The HTML export is a full workbench (click nodes for details, search, a SPARQL console). Embedding it via `srcdoc` makes it work inside the notebook output.

In [ ]:
import html as html_mod
from IPython.display import HTML

page = db.to_html(title="pipeline.yaml")
HTML(f'<iframe srcdoc="{html_mod.escape(page)}" width="100%" height="620" style="border:1px solid #ddd; border-radius:8px;"></iframe>')

## 10. The database is just a file

Everything above lives in `pipeline.yaml` — read it, diff it, commit it, hand it to an agent. The quotes the model was made to supply are in it too, so every row says which sentence it came from.

In [ ]:
print(open("pipeline.yaml").read())

## Next steps

- The same path without Python: `trikedb extract graph.yaml doc.md -o prompt.txt`, send it to a model, then `trikedb import graph.yaml answer.md --dry-run`
- Keep `graph.yaml` in your repo and tell your agent to read it before data work
- Or serve it as an ontology layer over MCP: `trikedb mcp graph.yaml` (stdio, for local agent sessions)
- Docs & source: https://github.com/RyutoYoda/trikedb